# Online Grooming Detection
SimCSE-Base-RoBERTa + SVM on PAN 2012

In [1]:
!pip install transformers sentence-transformers scikit-learn torch joblib
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dataset download

In [3]:
!wget https://zenodo.org/records/3713280/files/pan12-sexual-predator-identification-test-and-training.zip?download=1 -O pan12-sexual-predator-identification-test-and-training.zip
!unzip pan12-sexual-predator-identification-test-and-training.zip
!unzip pan12-sexual-predator-identification-training-corpus-2012-05-01.zip
!unzip pan12-sexual-predator-identification-test-corpus-2012-05-21.zip
!mkdir -p pan2012
!mv pan12-sexual-predator-identification-training-corpus-2012-05-01 pan2012/train
!mv pan12-sexual-predator-identification-test-corpus-2012-05-21 pan2012/test


--2026-06-30 16:40:47--  https://zenodo.org/records/3713280/files/pan12-sexual-predator-identification-test-and-training.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.52.235, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 91163031 (87M) [application/octet-stream]
Saving to: ‘pan12-sexual-predator-identification-test-and-training.zip’

pan12-sexual-predat 100%[===================>]  86.94M  9.17MB/s    in 82s     

2026-06-30 16:42:11 (1.06 MB/s) - ‘pan12-sexual-predator-identification-test-and-training.zip’ saved [91163031/91163031]

Archive:  pan12-sexual-predator-identification-test-and-training.zip
  inflating: pan12-sexual-predator-identification-test-corpus-2012-05-21.zip  
  inflating: pan12-sexual-predator-identification-training-corpus-2012-05-01.zip  
Archive:  pan12-sexual-predator-identification-training-corpus-2012-05-01.zip
  inflating: pan12-sex

## Config & Imports

In [4]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Tuple

import numpy as np
import joblib
import torch
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, classification_report
)

DATA_DIR       = Path("pan2012")
TRAIN_XML      = DATA_DIR / "train" / "pan12-sexual-predator-identification-training-corpus-2012-05-01.xml"
TRAIN_PRED_TXT = DATA_DIR / "train" / "pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt"
TEST_XML       = DATA_DIR / "test"  / "pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"
TEST_PRED_TXT  = DATA_DIR / "test"  / "pan12-sexual-predator-identification-groundtruth-problem1.txt"

CHECKPOINT_DIR = Path("/content/drive/MyDrive/grooming_detection/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME   = "princeton-nlp/sup-simcse-roberta-base"
MIN_MESSAGES = 7
BATCH_SIZE   = 64
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Checkpoint dir: {CHECKPOINT_DIR.resolve()}")

Device: cuda
Checkpoint dir: /content/drive/MyDrive/grooming_detection/checkpoints


## Data Loading & Preprocessing

In [5]:
def load_predator_ids(path: Path) -> set:
    if not path.exists():
        xml_path = path.with_suffix(".xml")
        if xml_path.exists():
            tree = ET.parse(xml_path)
            ids = {u.get("id") for u in tree.getroot().iter("user") if u.get("id")}
            print(f"Loaded {len(ids):,} predator IDs from {xml_path.name}")
            return ids
        raise FileNotFoundError(f"No predator ID file found at {path} or .xml")
    ids = set()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            pid = line.strip()
            if pid:
                ids.add(pid)
    print(f"Loaded {len(ids):,} predator IDs from {path.name}")
    return ids


def clean_text(text: str) -> str:
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    tokens = text.split()
    return " ".join(
        tok for tok in tokens
        if sum(1 for c in tok if ord(c) < 128) / max(len(tok), 1) >= 0.70
    ).strip()


def is_english(text: str) -> bool:
    if not text:
        return False
    return sum(1 for c in text if ord(c) < 128) / len(text) > 0.80


def load_conversations(xml_path: Path, predator_ids: set) -> Tuple[List[str], List[int]]:
    print(f"Parsing {xml_path.name} ...")
    root = ET.parse(xml_path).getroot()
    texts, labels = [], []
    skipped = {"msg": 0, "authors": 0, "lang": 0}

    for conv in root.iter("conversation"):
        messages = list(conv.iter("message"))

        if len(messages) < MIN_MESSAGES:
            skipped["msg"] += 1
            continue

        authors = {
            m.find("author").text.strip()
            for m in messages
            if m.find("author") is not None and m.find("author").text
        }
        if len(authors) != 2:
            skipped["authors"] += 1
            continue

        raw = " ".join(
            m.find("text").text.strip()
            for m in messages
            if m.find("text") is not None and m.find("text").text
        )
        cleaned = clean_text(raw)

        if not is_english(cleaned) or len(cleaned) < 10:
            skipped["lang"] += 1
            continue

        texts.append(cleaned)
        labels.append(1 if authors & predator_ids else 0)

    pos = sum(labels)
    print(f"  Kept {len(texts):,}  |  predatory={pos:,}  non-predatory={len(labels)-pos:,}")
    print(f"  Skipped — short: {skipped['msg']:,}  bad-authors: {skipped['authors']:,}  non-EN: {skipped['lang']:,}")
    return texts, labels


train_predators        = load_predator_ids(TRAIN_PRED_TXT)
train_texts, train_labels = load_conversations(TRAIN_XML, train_predators)

test_predators        = load_predator_ids(TEST_PRED_TXT)
test_texts, test_labels  = load_conversations(TEST_XML, test_predators)

y_train = np.array(train_labels)
y_test  = np.array(test_labels)

Loaded 142 predator IDs from pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt
Parsing pan12-sexual-predator-identification-training-corpus-2012-05-01.xml ...
  Kept 9,467  |  predatory=952  non-predatory=8,515
  Skipped — short: 50,064  bad-authors: 7,389  non-EN: 7
Loaded 254 predator IDs from pan12-sexual-predator-identification-groundtruth-problem1.txt
Parsing pan12-sexual-predator-identification-test-corpus-2012-05-17.xml ...
  Kept 21,716  |  predatory=1,698  non-predatory=20,018
  Skipped — short: 116,083  bad-authors: 17,323  non-EN: 6


## Encode with SimCSE-Base-RoBERTa

In [6]:
EMB_TRAIN_PATH = CHECKPOINT_DIR / "embeddings_train_simcse-base-roberta.npy"
EMB_TEST_PATH  = CHECKPOINT_DIR / "embeddings_test_simcse-base-roberta.npy"


def encode(texts: List[str], cache_path: Path, split: str) -> np.ndarray:
    if cache_path.exists():
        arr = np.load(cache_path)
        print(f"Loaded {split} embeddings from cache — shape {arr.shape}")
        return arr
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    print(f"Encoding {len(texts):,} {split} texts ...")
    arr = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    np.save(cache_path, arr)
    print(f"Saved {split} embeddings → {cache_path}")
    return arr


X_train = encode(train_texts, EMB_TRAIN_PATH, "train")
X_test  = encode(test_texts,  EMB_TEST_PATH,  "test")

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

Saved train embeddings → /content/drive/MyDrive/grooming_detection/checkpoints/embeddings_train_simcse-base-roberta.npy


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 21,716 test texts ...


Batches:   0%|          | 0/340 [00:00<?, ?it/s]

Saved test embeddings → /content/drive/MyDrive/grooming_detection/checkpoints/embeddings_test_simcse-base-roberta.npy
X_train: (9467, 768)  X_test: (21716, 768)


## Train SVM

In [7]:
CLF_PATH    = CHECKPOINT_DIR / "svm_simcse_base_roberta.joblib"
SCALER_PATH = CHECKPOINT_DIR / "scaler_simcse_base_roberta.joblib"


def train_svm(X_tr, y_tr):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    clf = SVC(
        kernel="rbf",
        C=1.0,
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    print("Training SVM ...")
    clf.fit(X_tr_s, y_tr)
    joblib.dump(scaler, SCALER_PATH)
    joblib.dump(clf,    CLF_PATH)
    print(f"Saved scaler → {SCALER_PATH}")
    print(f"Saved SVM    → {CLF_PATH}")
    return scaler, clf


if CLF_PATH.exists() and SCALER_PATH.exists():
    print("Loading SVM from cache ...")
    scaler = joblib.load(SCALER_PATH)
    clf    = joblib.load(CLF_PATH)
else:
    scaler, clf = train_svm(X_train, y_train)

Training SVM ...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Saved scaler → /content/drive/MyDrive/grooming_detection/checkpoints/scaler_simcse_base_roberta.joblib
Saved SVM    → /content/drive/MyDrive/grooming_detection/checkpoints/svm_simcse_base_roberta.joblib


## Evaluate

In [8]:
X_test_s = scaler.transform(X_test)
y_pred   = clf.predict(X_test_s)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
f05  = fbeta_score(y_test, y_pred, beta=0.5, zero_division=0)

print("SimCSE-Base-RoBERTa + SVM")
print("-" * 30)
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1        : {f1:.4f}")
print(f"F0.5      : {f05:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["non-predatory", "predatory"]))

SimCSE-Base-RoBERTa + SVM
------------------------------
Accuracy  : 0.9932
Precision : 0.9663
Recall    : 0.9464
F1        : 0.9563
F0.5      : 0.9623

               precision    recall  f1-score   support

non-predatory       1.00      1.00      1.00     20018
    predatory       0.97      0.95      0.96      1698

     accuracy                           0.99     21716
    macro avg       0.98      0.97      0.98     21716
 weighted avg       0.99      0.99      0.99     21716

